In [15]:
# Hệ thống Gợi ý Món Ăn với TAS-B Embedding

# Kiến trúc:
# 1. **Semi-structured NL State** → Generate Query (LLM)
# 2. **Query + Reviews** → TAS-B Embedding Space → Similarity Scores
# 3. **Top dishes** → Generate Recommendation & Explanation (LLM)

## 1. Setup & Load Data

In [16]:
import json
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load environment và config API
load_dotenv()
API_KEY = os.getenv("API_KEY")
genai.configure(api_key=API_KEY)
MODEL = "models/gemini-2.0-flash-lite"

print("✅ Đã load thư viện và config API")

✅ Đã load thư viện và config API


In [17]:
# Load dialog state
STATE_FILE = "../notebooks/dialog_state.json"
with open(STATE_FILE, "r", encoding="utf-8") as f:
    state = json.load(f)

# Load food dataset
FOOD_DATA_PATH = "../data/vnexpress_foods_detail_merged.csv"
df_foods = pd.read_csv(FOOD_DATA_PATH)

print(f"✅ Đã load dialog state")
print(f"✅ Đã load {len(df_foods)} món ăn từ dataset\n")
print("📄 Dialog State:")
print(json.dumps(state["hard_constraints"], indent=2, ensure_ascii=False))

✅ Đã load dialog state
✅ Đã load 893 món ăn từ dataset

📄 Dialog State:
{
  "type_of_food": [
    "món tết"
  ],
  "ingredients": [
    "nem chua"
  ],
  "cook_time": [
    "nhanh",
    "trung bình",
    "lâu"
  ],
  "algeric": [
    "none"
  ]
}


## 2. Filter Dishes by Hard Constraints (MUST Match)

In [18]:
import re

def filter_by_hard_constraints(df, state):
    """
    Lọc món ăn theo hard constraints (BẮT BUỘC phải match):
    - type_of_food: Loại món ăn
    - ingredients: Nguyên liệu PHẢI CÓ
    - cook_time: Thời gian nấu
    - algeric: Thành phần PHẢI TRÁNH
    
    Returns: filtered dataframe
    """
    filtered_df = df.copy()
    
    # 1. Type of food
    if state["hard_constraints"]["type_of_food"]:
        type_list = state["hard_constraints"]["type_of_food"]
        type_list_lower = [t.lower() for t in type_list]
        filtered_df = filtered_df[
            filtered_df['type_of_food'].str.lower().isin(type_list_lower)
        ]
    
    # 2. Ingredients (PHẢI CÓ)
    if state["hard_constraints"]["ingredients"]:
        ingredient_keywords = state["hard_constraints"]["ingredients"]
        if "everything" not in [item.lower() for item in ingredient_keywords]:
            def has_ingredient(row):
                ingredients_str = str(row['ingredients']).lower()
                title_str = str(row['title']).lower()
                full_text = ingredients_str + ' ' + title_str
                # Phải có TẤT CẢ nguyên liệu được yêu cầu
                for keyword in ingredient_keywords:
                    keyword_lower = keyword.lower()
                    if keyword_lower not in full_text:
                        return False
                return True
            filtered_df = filtered_df[filtered_df.apply(has_ingredient, axis=1)]
    
    # 3. Allergic (PHẢI TRÁNH)
    if state["hard_constraints"]["algeric"]:
        allergic_items = state["hard_constraints"]["algeric"]
        if "none" not in [item.lower() for item in allergic_items]:
            def is_safe(row):
                ingredients_str = str(row['ingredients']).lower()
                title_str = str(row['title']).lower()
                full_text = ingredients_str + ' ' + title_str
                return not any(allergen.lower() in full_text for allergen in allergic_items)
            filtered_df = filtered_df[filtered_df.apply(is_safe, axis=1)]
    
    # 4. Cook time
    if state["hard_constraints"]["cook_time"]:
        cook_time_prefs = [item.lower() for item in state["hard_constraints"]["cook_time"]]
        
        # Nếu có cả 3 loại thì không lọc (chấp nhận tất cả)
        if not (len(cook_time_prefs) == 3 and "nhanh" in cook_time_prefs and 
                "trung bình" in cook_time_prefs and "lâu" in cook_time_prefs):
            def match_cook_time(row):
                cook_time_str = str(row['cook_time']).lower()
                try:
                    time_match = re.search(r'(\d+)', cook_time_str)
                    if time_match:
                        minutes = int(time_match.group(1))
                        for pref in cook_time_prefs:
                            if "nhanh" in pref or "quick" in pref:
                                if minutes <= 30:
                                    return True
                            elif "trung bình" in pref or "medium" in pref:
                                if 30 < minutes <= 60:
                                    return True
                            elif "lâu" in pref or "long" in pref or "chậm" in pref:
                                if minutes > 60:
                                    return True
                except:
                    pass
                return False
            filtered_df = filtered_df[filtered_df.apply(match_cook_time, axis=1)]
    
    return filtered_df

# Apply hard constraints filtering
df_filtered = filter_by_hard_constraints(df_foods, state)

print(f"✅ Sau khi lọc theo hard constraints:")
print(f"   - Ban đầu: {len(df_foods)} món ăn")
print(f"   - Sau lọc: {len(df_filtered)} món ăn")

if len(df_filtered) == 0:
    print("\n⚠️ CẢNH BÁO: Không có món nào thỏa mãn tất cả hard constraints!")
    print("   → Sẽ dùng toàn bộ dataset để tính similarity (fallback)")
    df_filtered = df_foods.copy()
else:
    print(f"\n🎯 Danh sách các món thỏa mãn hard constraints:")
    for idx, row in df_filtered.head(10).iterrows():
        print(f"   - {row['title']} ({row['type_of_food']})")
        if pd.notna(row['ingredients']):
            print(f"     Nguyên liệu: {row['ingredients'][:100]}...")

✅ Sau khi lọc theo hard constraints:
   - Ban đầu: 893 món ăn
   - Sau lọc: 1 món ăn

🎯 Danh sách các món thỏa mãn hard constraints:
   - Cách làm gỏi cuốn thập cẩm phiên bản 'vét tủ lạnh' (Món Tết)
     Nguyên liệu: ['300 gr thịt lợn ba chỉ', '200 gr tôm tươi', '100 gr giò lụa', '100 gr nem chua', '4 bìa đậu phụ', ...


## 3. Generate Query from Semi-structured State (LLM)

In [19]:
def generate_query_from_state(state):
    """
    Sử dụng LLM để sinh ra natural language query từ dialog state
    """
    prompt = f"""
Dựa trên thông tin sở thích người dùng trong JSON sau, hãy tạo ra một câu query tự nhiên bằng tiếng Việt như thể người dùng đang hỏi về món ăn.

Dialog State:
{json.dumps(state, indent=2, ensure_ascii=False)}

Yêu cầu:
- Tạo câu query tự nhiên, giống cách người Việt nói chuyện hàng ngày
- Bao gồm TẤT CẢ thông tin từ hard_constraints (type_of_food, ingredients, cook_time, algeric)
- Nếu có soft_constraints (num_of_people, calories) thì cũng bao gồm
- Câu query phải rõ ràng, dễ hiểu

Ví dụ:
- Nếu state có type_of_food=["món tết"], ingredients=["nem chua"], cook_time=["nhanh"]
  → "Tôi muốn tìm món Tết có nem chua, nấu nhanh dưới 30 phút"

CHỈ trả về câu query, KHÔNG có giải thích thêm.
"""
    
    response = genai.GenerativeModel(MODEL).generate_content(prompt)
    return response.text.strip()

# Generate query
generated_query = generate_query_from_state(state)
print("🤖 Generated Query:")
print(f'"{generated_query}"')

🤖 Generated Query:
"Tôi muốn tìm món ăn ngày Tết có nem chua, nấu được trong thời gian nhanh, trung bình hoặc lâu, không có dị ứng gì cả."


## 4. Setup TAS-B Embedding Model

In [20]:
# Load TAS-B model (msmarco-distilbert-base-tas-b)
# TAS-B là model được train cho semantic search với query-document matching
print("⏳ Đang load TAS-B model...")
tasb_model = SentenceTransformer('sentence-transformers/msmarco-distilbert-base-tas-b')
print("✅ Đã load TAS-B model thành công!")

⏳ Đang load TAS-B model...
✅ Đã load TAS-B model thành công!
✅ Đã load TAS-B model thành công!


## 5. Create Embeddings for FILTERED Dishes (Reviews/Descriptions)

In [21]:
# Tạo "review-like" text cho mỗi món ăn từ description + type_of_food + cook_time
def create_dish_text(row):
    """
    Kết hợp thông tin món ăn thành text giống review để embedding
    """
    parts = []
    
    if pd.notna(row['type_of_food']):
        parts.append(f"{row['type_of_food']}")
    
    if pd.notna(row['title']):
        parts.append(f"{row['title']}")
    
    if pd.notna(row['description']):
        parts.append(f"{row['description']}")
    
    if pd.notna(row['cook_time']):
        parts.append(f"Thời gian nấu: {row['cook_time']}")
    
    return ". ".join(parts)

# Tạo text cho các món đã được lọc
df_filtered['review_text'] = df_filtered.apply(create_dish_text, axis=1)

print(f"✅ Đã tạo review text cho {len(df_filtered)} món ăn (sau lọc)")
print(f"\nVí dụ review text của món đầu tiên:")
print(f"'{df_filtered.iloc[0]['review_text'][:200]}...'")

✅ Đã tạo review text cho 1 món ăn (sau lọc)

Ví dụ review text của món đầu tiên:
'Món Tết. Cách làm gỏi cuốn thập cẩm phiên bản 'vét tủ lạnh'. Tận dụng thực phẩm dư thừa từ Tết, món gỏi cuốn thập cẩm nhiều màu sắc, tươi mát hấp dẫn nhiều sắc màu và giúp giản ngán hiệu quả....'


In [22]:
# Encode các món đã lọc thành embeddings
print(f"⏳ Đang tạo embeddings cho {len(df_filtered)} món ăn (đã lọc)...")
dish_embeddings = tasb_model.encode(
    df_filtered['review_text'].tolist(), 
    show_progress_bar=True,
    batch_size=32
)

print(f"✅ Đã tạo embeddings: {dish_embeddings.shape}")
print(f"   - {dish_embeddings.shape[0]} món ăn (thỏa mãn hard constraints)")
print(f"   - {dish_embeddings.shape[1]} dimensions")

⏳ Đang tạo embeddings cho 1 món ăn (đã lọc)...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.88it/s]

✅ Đã tạo embeddings: (1, 768)
   - 1 món ăn (thỏa mãn hard constraints)
   - 768 dimensions


## 6. Compute Query-Review Similarity

In [23]:
# Encode query
query_embedding = tasb_model.encode([generated_query])[0]

# Tính cosine similarity giữa query và các món đã lọc
similarities = cosine_similarity([query_embedding], dish_embeddings)[0]

# Thêm similarity scores vào dataframe
df_filtered['similarity_score'] = similarities

# Sort theo similarity giảm dần
df_ranked = df_filtered.sort_values('similarity_score', ascending=False)

print(f"✅ Đã tính similarity cho {len(df_filtered)} món ăn (thỏa mãn hard constraints)")
print(f"\n🎯 Query: '{generated_query}'")
print(f"\n📊 Top 5 món ăn có similarity cao nhất:")
print("="*80)
for idx, row in df_ranked.head(5).iterrows():
    print(f"Similarity: {row['similarity_score']:.3f} | {row['title']}")
    print(f"   Loại: {row['type_of_food']} | Thời gian: {row['cook_time']}")
    
    # Hiển thị nguyên liệu để verify
    if pd.notna(row['ingredients']):
        print(f"   Nguyên liệu: {row['ingredients'][:150]}...")
    
    if pd.notna(row['description']):
        print(f"   Mô tả: {row['description'][:100]}...")
    print()

✅ Đã tính similarity cho 1 món ăn (thỏa mãn hard constraints)

🎯 Query: 'Tôi muốn tìm món ăn ngày Tết có nem chua, nấu được trong thời gian nhanh, trung bình hoặc lâu, không có dị ứng gì cả.'

📊 Top 5 món ăn có similarity cao nhất:
Similarity: 0.917 | Cách làm gỏi cuốn thập cẩm phiên bản 'vét tủ lạnh'
   Loại: Món Tết | Thời gian: nan
   Nguyên liệu: ['300 gr thịt lợn ba chỉ', '200 gr tôm tươi', '100 gr giò lụa', '100 gr nem chua', '4 bìa đậu phụ', '4 quả trứng gà', 'Rau của quả (tùy chọn): Xà lách...
   Mô tả: Tận dụng thực phẩm dư thừa từ Tết, món gỏi cuốn thập cẩm nhiều màu sắc, tươi mát hấp dẫn nhiều sắc m...



## 7. Generate Recommendation & Explanation (LLM)

In [24]:
def generate_recommendation_explanation(query, top_dishes_df, top_k=3):
    """
    Generate personalized explanation cho top recommendations
    """
    # Lấy top K dishes
    top_dishes = top_dishes_df.head(top_k)
    
    # Tạo context cho LLM
    dishes_info = []
    for idx, row in top_dishes.iterrows():
        dishes_info.append({
            "title": row['title'],
            "type": row['type_of_food'],
            "cook_time": row['cook_time'],
            "description": row['description'],
            "similarity_score": float(row['similarity_score'])
        })
    
    prompt = f"""
Bạn là một chuyên gia ẩm thực nhiệt tình, giúp người dùng tìm món ăn phù hợp.

User Query: "{query}"

Top {top_k} món ăn phù hợp nhất (theo độ tương đồng):

{json.dumps(dishes_info, indent=2, ensure_ascii=False)}

Nhiệm vụ:
1. Giới thiệu ngắn gọn món ăn PHÙ HỢP NHẤT (món đầu tiên - similarity cao nhất)
2. Giải thích TẠI SAO món này phù hợp với yêu cầu của người dùng
3. Nêu điểm nổi bật của món (thời gian nấu, đặc điểm...)
4. Gợi ý 1-2 món khác trong danh sách nếu phù hợp

Yêu cầu:
- Viết bằng tiếng Việt, thân thiện, nhiệt tình
- Ngắn gọn, súc tích (3-5 câu cho món chính, 1-2 câu cho món phụ)
- Sử dụng emoji phù hợp
- Kết thúc bằng câu hỏi "Bạn có muốn xem công thức chi tiết không?"
"""
    
    response = genai.GenerativeModel(MODEL).generate_content(prompt)
    return response.text.strip()

# Generate explanation
explanation = generate_recommendation_explanation(generated_query, df_ranked, top_k=3)

print("="*80)
print("🤖 RECOMMENDATION & EXPLANATION")
print("="*80)
print(explanation)
print("="*80)

🤖 RECOMMENDATION & EXPLANATION
Chào bạn! Tết đến rồi, mình sẽ giúp bạn tìm món ăn ngon nha! 😊

Món ăn phù hợp nhất với yêu cầu của bạn là **Gỏi cuốn thập cẩm phiên bản "vét tủ lạnh"**! 🥳

**Tại sao lại là gỏi cuốn?** Vì món này cực kỳ linh hoạt, bạn có thể tận dụng nem chua (có trong yêu cầu), rau củ còn lại trong tủ lạnh để cuốn, rất phù hợp với không khí Tết. 🤩 Món này lại tươi mát, giúp cân bằng sau những bữa ăn nhiều dầu mỡ. Thời gian chuẩn bị lại cực nhanh, phù hợp cho những ngày bận rộn.

Nếu bạn thích, bạn có thể thử kết hợp gỏi cuốn với các món khác như nem chua rán để tăng thêm hương vị cho bữa ăn ngày Tết.

Bạn có muốn xem công thức chi tiết không? 😋


## 9. Complete Pipeline Function

In [26]:
def recommend_dishes_pipeline(state, df_foods, tasb_model, top_k=5, verbose=True):
    """
    Complete recommendation pipeline:
    1. Filter by hard constraints (MUST match)
    2. Generate query from state (LLM)
    3. Create embeddings for filtered dishes
    4. Compute similarity (TAS-B)
    5. Rank dishes
    6. Generate explanation (LLM)
    
    Returns: (query, top_dishes_df, explanation)
    """
    if verbose:
        print("🔄 Starting recommendation pipeline...")
        print("="*80)
    
    # Step 1: Filter by hard constraints
    df_filtered = filter_by_hard_constraints(df_foods, state)
    if verbose:
        print(f"\n1️⃣ Filtered by hard constraints: {len(df_foods)} → {len(df_filtered)} dishes")
    
    if len(df_filtered) == 0:
        print("⚠️ No dishes match hard constraints! Using full dataset.")
        df_filtered = df_foods.copy()
    
    # Step 2: Generate query
    query = generate_query_from_state(state)
    if verbose:
        print(f"\n2️⃣ Generated Query: '{query}'")
    
    # Step 3: Create embeddings for filtered dishes
    df_filtered['review_text'] = df_filtered.apply(create_dish_text, axis=1)
    dish_embeddings = tasb_model.encode(df_filtered['review_text'].tolist(), show_progress_bar=False)
    
    # Step 4: Encode query and compute similarity
    query_embedding = tasb_model.encode([query])[0]
    similarities = cosine_similarity([query_embedding], dish_embeddings)[0]
    
    # Step 5: Rank dishes
    df_filtered['similarity_score'] = similarities
    df_ranked = df_filtered.sort_values('similarity_score', ascending=False)
    
    if verbose:
        print(f"\n3️⃣ Computed similarity for {len(df_filtered)} dishes")
        print(f"\n📊 Top {min(top_k, len(df_ranked))} recommendations:")
        for i, row in df_ranked.head(top_k).iterrows():
            print(f"   {row['similarity_score']:.3f} | {row['title']}")
            if pd.notna(row['ingredients']):
                print(f"      Nguyên liệu: {row['ingredients'][:100]}...")
    
    # Step 6: Generate explanation
    explanation = generate_recommendation_explanation(query, df_ranked, top_k=min(3, top_k))
    
    if verbose:
        print(f"\n4️⃣ Generated explanation:")
        print("="*80)
        print(explanation)
        print("="*80)
    
    return query, df_ranked.head(top_k), explanation


# Test complete pipeline
print("\n🚀 TESTING COMPLETE PIPELINE\n")
query, top_dishes, explanation = recommend_dishes_pipeline(
    state, df_foods, tasb_model, top_k=5
)


🚀 TESTING COMPLETE PIPELINE

🔄 Starting recommendation pipeline...

1️⃣ Filtered by hard constraints: 893 → 1 dishes

2️⃣ Generated Query: '"Cho tôi xin món Tết nào có nem chua, nấu được cả nhanh, trung bình hoặc lâu, mà không có dị ứng gì nha."'

3️⃣ Computed similarity for 1 dishes

📊 Top 1 recommendations:
   0.911 | Cách làm gỏi cuốn thập cẩm phiên bản 'vét tủ lạnh'
      Nguyên liệu: ['300 gr thịt lợn ba chỉ', '200 gr tôm tươi', '100 gr giò lụa', '100 gr nem chua', '4 bìa đậu phụ', ...

2️⃣ Generated Query: '"Cho tôi xin món Tết nào có nem chua, nấu được cả nhanh, trung bình hoặc lâu, mà không có dị ứng gì nha."'

3️⃣ Computed similarity for 1 dishes

📊 Top 1 recommendations:
   0.911 | Cách làm gỏi cuốn thập cẩm phiên bản 'vét tủ lạnh'
      Nguyên liệu: ['300 gr thịt lợn ba chỉ', '200 gr tôm tươi', '100 gr giò lụa', '100 gr nem chua', '4 bìa đậu phụ', ...

4️⃣ Generated explanation:
Chào bạn! 🥰 Với yêu cầu của bạn, món ăn phù hợp nhất là:

1.  **Cách làm gỏi cuốn thập cẩm phiên

## 10. Nghiên Cứu Chunking Strategy - Phân Đoạn Thông Tin Món Ăn

Mỗi món ăn có nhiều loại thông tin:
- **Nguyên liệu**: ["200gr gà", "100gr sả", ...] 
- **Bước nấu**: ["Bước 1: ...", "Bước 2: ...", ...]
- **Mô tả**: Text mô tả món ăn

**Câu hỏi**: Nên embedding như thế nào?
- **Option 1 (Fine-grained)**: Mỗi nguyên liệu = 1 vector, mỗi bước = 1 vector
- **Option 2 (Coarse-grained)**: Tất cả nguyên liệu = 1 vector, tất cả bước = 1 vector  
- **Option 3 (Hybrid)**: Kết hợp theo nhóm logic

→ Mục tiêu: Mỗi món = **List of sentences** (như 3 màu trong sơ đồ)

In [33]:
# Load món ăn vừa được recommend (món có similarity cao nhất)
sample_dish = top_dishes.iloc[0]

print("📋 PHÂN TÍCH MÓN ĂN ĐƯỢC RECOMMEND")
print("="*80)
print(f"Tên món: {sample_dish['title']}")
print(f"Loại: {sample_dish['type_of_food']}")
print(f"Similarity score: {sample_dish['similarity_score']:.4f}")
print(f"\n🔍 CẤU TRÚC DỮ LIỆU:\n")

# 1. Ingredients
if pd.notna(sample_dish['ingredients']):
    ingredients_raw = sample_dish['ingredients']
    print(f"1️⃣ NGUYÊN LIỆU (raw):")
    print(f"   Type: {type(ingredients_raw)}")
    print(f"   Content: {ingredients_raw[:200]}...")
    
    # Parse nếu là string list
    if isinstance(ingredients_raw, str):
        try:
            import ast
            ingredients_list = ast.literal_eval(ingredients_raw)
            if isinstance(ingredients_list, list):
                print(f"\n   📦 Parsed thành list: {len(ingredients_list)} items")
                for i, item in enumerate(ingredients_list[:5], 1):
                    print(f"      [{i}] {item}")
                if len(ingredients_list) > 5:
                    print(f"      ... và {len(ingredients_list)-5} items khác")
        except:
            print("   ⚠️ Không parse được thành list")

# 2. Steps
if pd.notna(sample_dish['step']):
    steps_raw = sample_dish['step']
    print(f"\n2️⃣ BƯỚC NẤU (raw):")
    print(f"   Type: {type(steps_raw)}")
    print(f"   Content: {steps_raw[:200]}...")
    
    # Parse nếu là string list
    if isinstance(steps_raw, str):
        try:
            import ast
            steps_list = ast.literal_eval(steps_raw)
            if isinstance(steps_list, list):
                print(f"\n   📦 Parsed thành list: {len(steps_list)} bước")
                for i, step in enumerate(steps_list[:3], 1):
                    print(f"      [Bước {i}] {step[:100]}...")
                if len(steps_list) > 3:
                    print(f"      ... và {len(steps_list)-3} bước khác")
        except:
            print("   ⚠️ Không parse được thành list")

# 3. Description
if pd.notna(sample_dish['description']):
    print(f"\n3️⃣ MÔ TẢ:")
    print(f"   Type: {type(sample_dish['description'])}")
    print(f"   Length: {len(sample_dish['description'])} chars")
    print(f"   Content: {sample_dish['description'][:200]}...")

print("\n" + "="*80)

📋 PHÂN TÍCH MÓN ĂN ĐƯỢC RECOMMEND
Tên món: Cách làm gỏi cuốn thập cẩm phiên bản 'vét tủ lạnh'
Loại: Món Tết
Similarity score: 0.9112

🔍 CẤU TRÚC DỮ LIỆU:

1️⃣ NGUYÊN LIỆU (raw):
   Type: <class 'str'>
   Content: ['300 gr thịt lợn ba chỉ', '200 gr tôm tươi', '100 gr giò lụa', '100 gr nem chua', '4 bìa đậu phụ', '4 quả trứng gà', 'Rau của quả (tùy chọn): Xà lách, rau mùi, cà rốt, dưa chuột, dứa, khế, củ đâu...'...

   📦 Parsed thành list: 9 items
      [1] 300 gr thịt lợn ba chỉ
      [2] 200 gr tôm tươi
      [3] 100 gr giò lụa
      [4] 100 gr nem chua
      [5] 4 bìa đậu phụ
      ... và 4 items khác

2️⃣ BƯỚC NẤU (raw):
   Type: <class 'str'>
   Content: ['Bước 1: Trứng gà đánh tan cùng chút gia vị, thoa chút dầu ăn đều khắp mặt chảo rồi cho từng ít vào láng đều, rán ở lửa nhỏ vừa cho trứng mịn màng, chín đều. Nếu thích tăng màu sắc thì tách lòng đỏ v...

   📦 Parsed thành list: 7 bước
      [Bước 1] Bước 1: Trứng gà đánh tan cùng chút gia vị, thoa chút dầu ăn đều khắp mặt chảo rồi 

### 10.1 Chunking Strategies - 3 Cách Phân Đoạn

In [34]:
import ast

def parse_list_field(field_value):
    """Parse string representation of list to actual list"""
    if pd.isna(field_value):
        return []
    if isinstance(field_value, list):
        return field_value
    if isinstance(field_value, str):
        try:
            return ast.literal_eval(field_value)
        except:
            return [field_value]
    return []


def strategy_1_fine_grained(row):
    """
    STRATEGY 1: Fine-grained (Chi tiết)
    - Mỗi nguyên liệu = 1 sentence
    - Mỗi bước nấu = 1 sentence
    - Mô tả = 1 sentence
    - Metadata (type, cook_time) = 1 sentence
    
    → Nhiều vector nhỏ, chi tiết cao
    """
    sentences = []
    
    # Metadata
    if pd.notna(row['type_of_food']):
        sentences.append(f"Đây là món {row['type_of_food']}")
    
    if pd.notna(row['title']):
        sentences.append(f"Tên món: {row['title']}")
    
    if pd.notna(row['cook_time']):
        sentences.append(f"Thời gian nấu: {row['cook_time']}")
    
    # Nguyên liệu - MỖI ITEM = 1 SENTENCE
    ingredients = parse_list_field(row['ingredients'])
    for ingredient in ingredients:
        sentences.append(f"Nguyên liệu: {ingredient}")
    
    # Bước nấu - MỖI BƯỚC = 1 SENTENCE
    steps = parse_list_field(row['step'])
    for i, step in enumerate(steps, 1):
        sentences.append(f"Bước {i}: {step}")
    
    # Mô tả
    if pd.notna(row['description']):
        sentences.append(f"Mô tả: {row['description']}")
    
    return sentences


def strategy_2_coarse_grained(row):
    """
    STRATEGY 2: Coarse-grained (Thô)
    - TẤT CẢ nguyên liệu = 1 sentence
    - TẤT CẢ bước nấu = 1 sentence
    - Mô tả = 1 sentence
    - Metadata = 1 sentence
    
    → Ít vector lớn, tổng quan
    """
    sentences = []
    
    # Metadata - GỘP CHUNG
    meta_parts = []
    if pd.notna(row['type_of_food']):
        meta_parts.append(f"Loại: {row['type_of_food']}")
    if pd.notna(row['title']):
        meta_parts.append(f"Tên: {row['title']}")
    if pd.notna(row['cook_time']):
        meta_parts.append(f"Thời gian: {row['cook_time']}")
    if meta_parts:
        sentences.append(". ".join(meta_parts))
    
    # Nguyên liệu - GỘP TẤT CẢ
    ingredients = parse_list_field(row['ingredients'])
    if ingredients:
        ingredients_text = ", ".join(ingredients)
        sentences.append(f"Nguyên liệu: {ingredients_text}")
    
    # Bước nấu - GỘP TẤT CẢ
    steps = parse_list_field(row['step'])
    if steps:
        steps_text = " → ".join([f"Bước {i}: {step}" for i, step in enumerate(steps, 1)])
        sentences.append(f"Cách làm: {steps_text}")
    
    # Mô tả
    if pd.notna(row['description']):
        sentences.append(f"Mô tả: {row['description']}")
    
    return sentences


def strategy_3_hybrid(row):
    """
    STRATEGY 3: Hybrid (Kết hợp)
    - Metadata = 1 sentence
    - Nguyên liệu theo NHÓM (chính/phụ) = 2-3 sentences
    - Bước nấu theo PHASE (chuẩn bị/nấu/hoàn thiện) = 3-4 sentences  
    - Mô tả = 1 sentence
    
    → Cân bằng giữa chi tiết và tổng quan
    """
    sentences = []
    
    # 1. Metadata
    meta_parts = []
    if pd.notna(row['type_of_food']):
        meta_parts.append(row['type_of_food'])
    if pd.notna(row['title']):
        meta_parts.append(row['title'])
    if pd.notna(row['cook_time']):
        meta_parts.append(f"thời gian {row['cook_time']}")
    if meta_parts:
        sentences.append(". ".join(meta_parts))
    
    # 2. Nguyên liệu - NHÓM 5 ITEMS/SENTENCE
    ingredients = parse_list_field(row['ingredients'])
    if ingredients:
        chunk_size = 5
        for i in range(0, len(ingredients), chunk_size):
            chunk = ingredients[i:i+chunk_size]
            sentences.append("Nguyên liệu: " + ", ".join(chunk))
    
    # 3. Bước nấu - NHÓM 3 BƯỚC/SENTENCE
    steps = parse_list_field(row['step'])
    if steps:
        chunk_size = 3
        for i in range(0, len(steps), chunk_size):
            chunk = steps[i:i+chunk_size]
            step_text = " → ".join([f"Bước {i+j+1}: {s}" for j, s in enumerate(chunk)])
            sentences.append(step_text)
    
    # 4. Mô tả
    if pd.notna(row['description']):
        sentences.append(row['description'])
    
    return sentences


# Test 3 strategies trên món được recommend
print("🧪 TEST 3 CHUNKING STRATEGIES\n")
print("="*80)

# Sử dụng sample_dish đã được load ở cell trước (món được recommend)
print(f"Món được recommend: {sample_dish['title']}")
# Kiểm tra xem có similarity_score không (có thể bị mất khi copy row)
if 'similarity_score' in sample_dish:
    print(f"Similarity score: {sample_dish['similarity_score']:.4f}\n")
else:
    print(f"(Món này được recommend từ pipeline)\n")

# Strategy 1
sentences_1 = strategy_1_fine_grained(sample_dish)
print(f"📌 STRATEGY 1: Fine-grained")
print(f"   → {len(sentences_1)} sentences")
print(f"   Ví dụ:")
for i, sent in enumerate(sentences_1[:5], 1):
    print(f"      [{i}] {sent[:80]}...")
if len(sentences_1) > 5:
    print(f"      ... và {len(sentences_1)-5} sentences khác")

print("\n" + "-"*80 + "\n")

# Strategy 2
sentences_2 = strategy_2_coarse_grained(sample_dish)
print(f"📌 STRATEGY 2: Coarse-grained")
print(f"   → {len(sentences_2)} sentences")
print(f"   Ví dụ:")
for i, sent in enumerate(sentences_2[:5], 1):
    print(f"      [{i}] {sent[:80]}...")

print("\n" + "-"*80 + "\n")

# Strategy 3
sentences_3 = strategy_3_hybrid(sample_dish)
print(f"📌 STRATEGY 3: Hybrid")
print(f"   → {len(sentences_3)} sentences")
print(f"   Ví dụ:")
for i, sent in enumerate(sentences_3[:5], 1):
    print(f"      [{i}] {sent[:80]}...")
if len(sentences_3) > 5:
    print(f"      ... và {len(sentences_3)-5} sentences khác")

print("\n" + "="*80)

🧪 TEST 3 CHUNKING STRATEGIES

Món được recommend: Cách làm gỏi cuốn thập cẩm phiên bản 'vét tủ lạnh'
Similarity score: 0.9112

📌 STRATEGY 1: Fine-grained
   → 19 sentences
   Ví dụ:
      [1] Đây là món Món Tết...
      [2] Tên món: Cách làm gỏi cuốn thập cẩm phiên bản 'vét tủ lạnh'...
      [3] Nguyên liệu: 300 gr thịt lợn ba chỉ...
      [4] Nguyên liệu: 200 gr tôm tươi...
      [5] Nguyên liệu: 100 gr giò lụa...
      ... và 14 sentences khác

--------------------------------------------------------------------------------

📌 STRATEGY 2: Coarse-grained
   → 4 sentences
   Ví dụ:
      [1] Loại: Món Tết. Tên: Cách làm gỏi cuốn thập cẩm phiên bản 'vét tủ lạnh'...
      [2] Nguyên liệu: 300 gr thịt lợn ba chỉ, 200 gr tôm tươi, 100 gr giò lụa, 100 gr nem...
      [3] Cách làm: Bước 1: Bước 1: Trứng gà đánh tan cùng chút gia vị, thoa chút dầu ăn đ...
      [4] Mô tả: Tận dụng thực phẩm dư thừa từ Tết, món gỏi cuốn thập cẩm nhiều màu sắc, t...

--------------------------------------------

### 10.2 Tạo Multi-vector Representation cho TẤT CẢ món ăn

In [35]:
# Áp dụng 3 strategies cho toàn bộ dataset
print("⏳ Đang tạo multi-vector representations cho tất cả món ăn...")
print("="*80)

# Strategy 1: Fine-grained
print("\n1️⃣ Strategy 1 (Fine-grained)...")
dish_sentences_s1 = []
for idx, row in df_foods.iterrows():
    sentences = strategy_1_fine_grained(row)
    dish_sentences_s1.append(sentences)

total_sentences_s1 = sum(len(s) for s in dish_sentences_s1)
avg_sentences_s1 = total_sentences_s1 / len(dish_sentences_s1)
print(f"   ✅ {len(dish_sentences_s1)} món ăn")
print(f"   ✅ {total_sentences_s1} sentences tổng")
print(f"   ✅ Trung bình: {avg_sentences_s1:.1f} sentences/món")

# Strategy 2: Coarse-grained
print("\n2️⃣ Strategy 2 (Coarse-grained)...")
dish_sentences_s2 = []
for idx, row in df_foods.iterrows():
    sentences = strategy_2_coarse_grained(row)
    dish_sentences_s2.append(sentences)

total_sentences_s2 = sum(len(s) for s in dish_sentences_s2)
avg_sentences_s2 = total_sentences_s2 / len(dish_sentences_s2)
print(f"   ✅ {len(dish_sentences_s2)} món ăn")
print(f"   ✅ {total_sentences_s2} sentences tổng")
print(f"   ✅ Trung bình: {avg_sentences_s2:.1f} sentences/món")

# Strategy 3: Hybrid
print("\n3️⃣ Strategy 3 (Hybrid)...")
dish_sentences_s3 = []
for idx, row in df_foods.iterrows():
    sentences = strategy_3_hybrid(row)
    dish_sentences_s3.append(sentences)

total_sentences_s3 = sum(len(s) for s in dish_sentences_s3)
avg_sentences_s3 = total_sentences_s3 / len(dish_sentences_s3)
print(f"   ✅ {len(dish_sentences_s3)} món ăn")
print(f"   ✅ {total_sentences_s3} sentences tổng")
print(f"   ✅ Trung bình: {avg_sentences_s3:.1f} sentences/món")

print("\n" + "="*80)
print("📊 SO SÁNH:")
print(f"   Strategy 1 (Fine):    {avg_sentences_s1:.1f} sentences/món → Chi tiết cao")
print(f"   Strategy 2 (Coarse):  {avg_sentences_s2:.1f} sentences/món → Tổng quan")
print(f"   Strategy 3 (Hybrid):  {avg_sentences_s3:.1f} sentences/món → Cân bằng")
print("="*80)

⏳ Đang tạo multi-vector representations cho tất cả món ăn...

1️⃣ Strategy 1 (Fine-grained)...
   ✅ 893 món ăn
   ✅ 16348 sentences tổng
   ✅ Trung bình: 18.3 sentences/món

2️⃣ Strategy 2 (Coarse-grained)...
   ✅ 893 món ăn
   ✅ 3570 sentences tổng
   ✅ Trung bình: 4.0 sentences/món

3️⃣ Strategy 3 (Hybrid)...
   ✅ 893 món ăn
   ✅ 5786 sentences tổng
   ✅ Trung bình: 6.5 sentences/món

📊 SO SÁNH:
   Strategy 1 (Fine):    18.3 sentences/món → Chi tiết cao
   Strategy 2 (Coarse):  4.0 sentences/món → Tổng quan
   Strategy 3 (Hybrid):  6.5 sentences/món → Cân bằng


### 10.3 Encode Multi-vectors - Mỗi món → List of Embeddings

In [36]:
def encode_multi_vector_dishes(dish_sentences_list, tasb_model, batch_size=32):
    """
    Encode tất cả sentences của tất cả món ăn thành vectors
    
    Input: dish_sentences_list = [
        ["sentence1 of dish1", "sentence2 of dish1", ...],  # Món 1
        ["sentence1 of dish2", "sentence2 of dish2", ...],  # Món 2
        ...
    ]
    
    Output: dish_embeddings_list = [
        np.array([[vec1], [vec2], ...]),  # List vectors của món 1
        np.array([[vec1], [vec2], ...]),  # List vectors của món 2
        ...
    ]
    """
    # Flatten tất cả sentences để encode 1 lần
    all_sentences = []
    sentence_counts = []  # Track số sentences của mỗi món
    
    for sentences in dish_sentences_list:
        all_sentences.extend(sentences)
        sentence_counts.append(len(sentences))
    
    print(f"⏳ Encoding {len(all_sentences)} sentences tổng...")
    
    # Encode tất cả sentences
    all_embeddings = tasb_model.encode(
        all_sentences, 
        show_progress_bar=True,
        batch_size=batch_size
    )
    
    # Split embeddings theo từng món
    dish_embeddings_list = []
    start_idx = 0
    for count in sentence_counts:
        end_idx = start_idx + count
        dish_embeddings = all_embeddings[start_idx:end_idx]
        dish_embeddings_list.append(dish_embeddings)
        start_idx = end_idx
    
    print(f"✅ Đã encode xong!")
    print(f"   → {len(dish_embeddings_list)} món ăn")
    print(f"   → Mỗi món có {sentence_counts[0]}-{max(sentence_counts)} vectors")
    
    return dish_embeddings_list


# Encode Strategy 3 (Hybrid) - RECOMMENDED
print("🚀 ENCODING STRATEGY 3 (HYBRID)\n")
print("="*80)

dish_embeddings_multi = encode_multi_vector_dishes(
    dish_sentences_s3, 
    tasb_model, 
    batch_size=64
)

# Kiểm tra kết quả
print(f"\n📊 KẾT QUẢ:")
print(f"   - Số món: {len(dish_embeddings_multi)}")
print(f"   - Món đầu tiên có {len(dish_embeddings_multi[0])} vectors")
print(f"   - Mỗi vector có {dish_embeddings_multi[0].shape[1]} dimensions")

print(f"\n📌 Ví dụ món đầu tiên:")
print(f"   Tên: {df_foods.iloc[0]['title']}")
print(f"   Số sentences: {len(dish_sentences_s3[0])}")
print(f"   Số vectors: {len(dish_embeddings_multi[0])}")
print(f"   Shape mỗi vector: {dish_embeddings_multi[0][0].shape}")

print("\n" + "="*80)

🚀 ENCODING STRATEGY 3 (HYBRID)

⏳ Encoding 5786 sentences tổng...


Batches: 100%|██████████| 91/91 [05:13<00:00,  3.44s/it]

✅ Đã encode xong!
   → 893 món ăn
   → Mỗi món có 5-23 vectors

📊 KẾT QUẢ:
   - Số món: 893
   - Món đầu tiên có 5 vectors
   - Mỗi vector có 768 dimensions

📌 Ví dụ món đầu tiên:
   Tên: Cách muối dưa hành truyền thống
   Số sentences: 5
   Số vectors: 5
   Shape mỗi vector: (768,)



### 10.4 Visualization - So sánh số lượng vectors của 3 strategies

In [ ]:
# In ra kết quả chi tiết của 3 strategies
print("📊 KẾT QUẢ CHI TIẾT CỦA 3 CHUNKING STRATEGIES")
print("="*80)

strategies = [
    ("Strategy 1: Fine-grained", dish_sentences_s1, avg_sentences_s1),
    ("Strategy 2: Coarse-grained", dish_sentences_s2, avg_sentences_s2),
    ("Strategy 3: Hybrid", dish_sentences_s3, avg_sentences_s3)
]

for idx, (title, dish_sentences, avg) in enumerate(strategies, 1):
    sentence_counts = [len(sentences) for sentences in dish_sentences]
    
    print(f"\n{idx}. {title}")
    print("-" * 80)
    print(f"   Tổng số món:           {len(dish_sentences):,}")
    print(f"   Tổng số sentences:     {sum(sentence_counts):,}")
    print(f"   Trung bình:            {avg:.1f} sentences/món")
    print(f"   Min sentences/món:     {min(sentence_counts)}")
    print(f"   Max sentences/món:     {max(sentence_counts)}")
    print(f"   Median sentences/món:  {np.median(sentence_counts):.1f}")
    
    # Hiển thị ví dụ món được recommend
    # Tìm index của món được recommend trong df_foods
    recommended_dish_title = top_dishes.iloc[0]['title']
    example_idx = df_foods[df_foods['title'] == recommended_dish_title].index[0]
    
    print(f"\n   📌 Ví dụ món được recommend: '{df_foods.iloc[example_idx]['title']}':")
    print(f"      Số sentences: {len(dish_sentences[example_idx])}")
    print(f"      Sentences:")
    for i, sent in enumerate(dish_sentences[example_idx], 1):
        print(f"         [{i}] {sent[:100]}...")

print("\n" + "="*80)
print("📊 SO SÁNH TÓM TẮT:")
print("="*80)
print(f"Strategy 1 (Fine):    {avg_sentences_s1:.1f} sentences/món")
print(f"   → ✅ Ưu điểm: Chi tiết cao, phù hợp search nguyên liệu/bước cụ thể")
print(f"   → ❌ Nhược điểm: Nhiều vectors → Chi phí tính toán cao")
print()
print(f"Strategy 2 (Coarse):  {avg_sentences_s2:.1f} sentences/món")
print(f"   → ✅ Ưu điểm: Ít vectors → Nhanh, hiệu quả")
print(f"   → ❌ Nhược điểm: Mất thông tin chi tiết, semantic quá chung")
print()
print(f"Strategy 3 (Hybrid):  {avg_sentences_s3:.1f} sentences/món")
print(f"   → ✅ Ưu điểm: Cân bằng giữa chi tiết và hiệu quả")
print(f"   → ✅ RECOMMENDED cho production!")
print("="*80)

print("\n🎯 KẾT QUẢ CUỐI CÙNG - STRATEGY 3 (HYBRID):")
print("="*80)
print(f"Mỗi món trong dataset được biểu diễn bằng:")
print(f"   - Trung bình {avg_sentences_s3:.1f} sentences")
print(f"   - Mỗi sentence = 1 vector 768-dim (TAS-B)")
print(f"   - Tổng cộng: {len(dish_embeddings_multi):,} món × ~{avg_sentences_s3:.0f} vectors = ~{len(dish_embeddings_multi) * avg_sentences_s3:,.0f} vectors")
print()
print("Cấu trúc mỗi món:")
print("   [Metadata] → [Nguyên liệu nhóm 1] → [Nguyên liệu nhóm 2] → ...")
print("   → [Bước 1-3] → [Bước 4-6] → ... → [Mô tả]")
print("="*80)

📊 KẾT QUẢ CHI TIẾT CỦA 3 CHUNKING STRATEGIES

1. Strategy 1: Fine-grained
--------------------------------------------------------------------------------
   Tổng số món:           893
   Tổng số sentences:     16,348
   Trung bình:            18.3 sentences/món
   Min sentences/món:     5
   Max sentences/món:     74
   Median sentences/món:  17.0

   📌 Ví dụ món 'Cách muối dưa hành truyền thống':
      Số sentences: 14
      Sentences:
         [1] Đây là món Món Tết...
         [2] Tên món: Cách muối dưa hành truyền thống...
         [3] Thời gian nấu: 45 phút...
         [4] Nguyên liệu: 1 kg hành củ tươi...
         [5] Nguyên liệu: Tro bếp hoặc nước vo gọa...
         [6] Nguyên liệu: Muối hạt, đường...
         [7] Nguyên liệu: Cà rốt trang trí (tùy chọn)...
         [8] Nguyên liệu: Lọ sạch...
         [9] Bước 1: Bước 1: Chọn hành củ: Nên chọn hành củ ta bánh tẻ, vừa phải, cầm chắc tay, tròn căng mọng, m...
         [10] Bước 2: Bước 2: Ngâm khử mùi hăng của hành: Theo kinh ng

### 10.5 Kết luận - Mỗi món = List of Sentences (3 màu trong sơ đồ)

Sau bước này, mỗi món ăn được biểu diễn bằng:

```
Món 1: [
    "Sentence 1 - Metadata (màu xanh)",
    "Sentence 2 - Nguyên liệu nhóm 1 (màu đỏ)", 
    "Sentence 3 - Nguyên liệu nhóm 2 (màu đỏ)",
    "Sentence 4 - Bước 1-3 (màu vàng)",
    "Sentence 5 - Bước 4-6 (màu vàng)",
    ...
]
```

→ Mỗi sentence sẽ được encode thành 1 vector 768-dim  
→ Query sẽ được so sánh với **TẤT CẢ** sentences của mỗi món  
→ Score cuối = **MaxSim** hoặc **AvgSim** của top-k sentences match nhất